In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy
import skimage
from scipy.ndimage import rotate
from scipy.signal import argrelmin
from sklearn.cluster import DBSCAN


def bump(x):
    return np.where(np.abs(x) < 1, np.exp(1 / (x ** 2 - 1) + 1) ** 0.2, 0)


def argrelmin_rotated(arr, angle, axis=0, order=1,
                      reshape=True, interp_order=1,
                      mode='constant', cval=np.inf):
    arr = np.asarray(arr)
    M, N = arr.shape

    arr_rot = rotate(arr, -angle,
                     reshape=reshape,
                     order=interp_order,
                     mode=mode,
                     cval=cval)
    M_rot, N_rot = arr_rot.shape

    rows_rot, cols_rot = argrelmin(arr_rot, axis=axis, order=order)

    center_orig = np.array([(M - 1) / 2.0, (N - 1) / 2.0])
    center_rot = np.array([(M_rot - 1) / 2.0, (N_rot - 1) / 2.0])

    theta = np.deg2rad(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    dr = rows_rot - center_rot[0]
    dc = cols_rot - center_rot[1]

    col_orig = cos_t * dc + sin_t * dr + center_orig[1]
    row_orig = -sin_t * dc + cos_t * dr + center_orig[0]

    row_idx = np.round(row_orig).astype(int)
    col_idx = np.round(col_orig).astype(int)
    valid = (
            (row_idx >= 0) & (row_idx < M) &
            (col_idx >= 0) & (col_idx < N)
    )
    return row_idx[valid], col_idx[valid]


def extract_df(I, r_v, theta_v, sigma_small=25, sigma_large=150, angle=60, poly_order=12, spline_lambda=1e-7,
               e_scale=1):
    fft = np.fft.fft(I, axis=1)
    tot = np.abs(fft[:, 0])
    recovered = skimage.filters.difference_of_gaussians(I, sigma_small, sigma_large, mode='wrap')

    args_r = argrelmin_rotated(recovered, angle=angle, axis=1, order=1)
    r_vals, theta_vals = r_v[args_r[0]], theta_v[args_r[1]]

    mask = (r_vals > 0.16) & (r_vals < 0.3) & (theta_vals > 0) & (theta_vals < 2)
    r_min, theta_min = r_vals[mask], theta_vals[mask]

    labels = DBSCAN(eps=0.1, min_samples=1).fit_predict(np.vstack((r_min * 15, theta_min)).T)
    sel = labels == np.argmax(np.bincount(labels))
    r_min, theta_min = r_min[sel], theta_min[sel]

    df_pts = pd.DataFrame({'r': r_min, 'theta': theta_min})
    df_pts = df_pts.groupby('theta', as_index=False)['r'].min()
    r_min, theta_min = df_pts['r'].values, df_pts['theta'].values

    crs = np.max(I, axis=1) - np.min(I, axis=1)
    idxs = np.intersect1d(theta_v, theta_min, return_indices=True)[1]
    cr = crs[idxs]

    E_min = r_min ** 2 / 2
    coeffs = np.polyfit(E_min, theta_min, poly_order, w=cr)
    r_sm = np.linspace(r_min.min(), r_min.max(), len(r_min))
    E = r_sm ** 2 / 2
    theta_sm = np.poly1d(coeffs)(E)
    group_delay = np.gradient(theta_sm, E * e_scale) * 2

    tot_sm = scipy.interpolate.make_smoothing_spline(r_v ** 2 / 2, tot, lam=spline_lambda)(E)

    return pd.DataFrame({
        'pr': r_min,
        'E': E * e_scale,
        'theta': theta_sm,
        'group_delay': group_delay,
        'signal': tot_sm / np.max(tot_sm)
    })

In [ ]:
from pathlib import Path

# List of experimental data files to process
exp_files = [
    r"J:\ctgroup\Edward\DATA\VMI\20220613\xe011_e_c.h5",
    r"J:\ctgroup\Edward\DATA\VMI\20220613\xe012_e_c.h5",
    # r"J:\ctgroup\Edward\DATA\VMI\20220613\xe013_e_c.h5",
    r"J:\ctgroup\Edward\DATA\VMI\20220613\xe005_e_c.h5",
]

ellipticities = ["ε=0.3", "ε=0.5", "ε=0.6"]
RES = 1200
all_datasets = []

for fname in exp_files:
    base_data = pd.read_hdf(fname, key="data")
    data = (base_data
            .query("sqrt(px**2+py**2+pz**2)<0.4")
            .query("abs(py)<0.5")
            .query("pz>0")
            .reset_index(drop=True))
    data_sym = data.copy()
    data_sym[['px', 'py', 'pz']] *= -1
    data = pd.concat([data, data_sym], ignore_index=True)

    data['pr'] = np.linalg.norm(data[['px', 'py', 'pz']].values, axis=1)
    data['phi'] = np.arctan2(data['px'], data['pz'])

    h2d, re_edges, pe_edges = np.histogram2d(data['pr'], data['phi'], bins=RES)
    r_vals = 0.5 * (re_edges[:-1] + re_edges[1:])
    theta_vals = 0.5 * (pe_edges[:-1] + pe_edges[1:])
    I_exp = (h2d.T * bump((r_vals - 0.25) / 0.1)).T

    all_datasets.append({
        'I': I_exp,
        'r_vals': r_vals,
        'theta_vals': theta_vals,
        'name': Path(fname).stem,
    })

In [ ]:
scale = 0.874 ** 2

all_exp_dfs = []

for dataset in all_datasets:
    I_exp = dataset['I']
    r_vals = dataset['r_vals']
    theta_vals = dataset['theta_vals']

    f_exp = px.imshow(I_exp, x=theta_vals, y=r_vals, origin='lower', aspect='auto').update_layout(
            title=f"Experimental Interlocking Error Bars — {dataset['name']}",
            yaxis_title="Momentum (pr)",
            xaxis_title="Angle (phi)"
    )
    f2_exp = go.Figure()
    exp_dfs = []

    for angle in np.arange(18, 81, 5):
        df = extract_df(I_exp, r_vals, theta_vals, angle=angle, e_scale=scale)
        f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'{angle}°')
        f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'{angle}°')
        exp_dfs.append(df)

    for hs in np.arange(100, 200, 6):
        df = extract_df(I_exp, r_vals, theta_vals, sigma_large=hs, e_scale=scale)
        f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'hs={hs}')
        f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'hs={hs}')
        f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'hs={hs}')
        f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'hs={hs}')
        exp_dfs.append(df)

    for ls in np.arange(10, 50, 4):
        df = extract_df(I_exp, r_vals, theta_vals, sigma_small=ls, e_scale=scale)
        f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'ls={ls}')
        f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'ls={ls}')
        exp_dfs.append(df)

    f_exp.show()
    f2_exp.update_layout(
            title=f"Experimental Group Delay vs Energy — {dataset['name']}",
            yaxis_title="Group Delay",
            xaxis_title="Energy (E)",
            yaxis_range=[0, 250],
            xaxis_range=[0.015, 0.035],
    )#.show()
    all_exp_dfs.append({'name': dataset['name'], 'dfs': exp_dfs})

# Flatten all dfs for combined statistics
exp_dfs_combined = [df for dataset in all_exp_dfs for df in dataset['dfs']]

In [ ]:
from matplotlib import colors as mcolors

ENERGY_CONV = 27.2114
TIME_CONV_FS = 0.02418884
num_points = 100
angular_shift = 0.6035 - np.radians(10)

all_E = np.hstack([df['E'].values for ds in all_exp_dfs for df in ds['dfs']])
E_min, E_max = all_E.min(), all_E.max()
E_grid = np.linspace(E_min, E_max, num_points)
E_ev_grid = E_grid * ENERGY_CONV

_palette = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink']

dataset_stats = []
for i, ds in enumerate(all_exp_dfs):
    gd = np.vstack([np.interp(E_grid, df['E'], df['group_delay']) for df in ds['dfs']])
    th = np.vstack([np.interp(E_grid, df['E'], df['theta']) for df in ds['dfs']])
    dataset_stats.append({
        'name': ds['name'],
        'gd_mean': gd.mean(axis=0), 'gd_std': gd.std(axis=0),
        'th_mean': th.mean(axis=0), 'th_std': th.std(axis=0),
        'color': _palette[i % len(_palette)],
    })

# Group delay vs Energy
fig1 = go.Figure()
for ds in dataset_stats:
    c = ds['color']
    rgba = f"rgba{mcolors.to_rgba(c, alpha=0.2)}"
    fig1.add_trace(go.Scatter(x=E_ev_grid, y=(ds['gd_mean'] + ds['gd_std']) * TIME_CONV_FS,
                              line=dict(color=c, width=0), showlegend=False, hoverinfo='skip'))
    fig1.add_trace(go.Scatter(x=E_ev_grid, y=(ds['gd_mean'] - ds['gd_std']) * TIME_CONV_FS,
                              fill='tonexty', fillcolor=rgba,
                              line=dict(color=c, width=0), showlegend=False, hoverinfo='skip'))
    fig1.add_trace(go.Scatter(x=E_ev_grid, y=ds['gd_mean'] * TIME_CONV_FS,
                              mode='lines', name=ds['name'], line_color=c))

fig1.add_hline(y=2.67, line_color='black', line_dash='dash',
               annotation_text="Laser Period", annotation_position="top left")
fig1.update_layout(title="Group Delay vs Energy", xaxis_title="Energy (eV)", yaxis_title="Group Delay (fs)",
                   xaxis_range=[0.3, 0.9], yaxis_range=[0, 5])
fig1.show()

# Phase vs Energy
fig2 = go.Figure()
for ds in dataset_stats:
    c = ds['color']
    rgba = f"rgba{mcolors.to_rgba(c, alpha=0.2)}"
    exp_phase = ds['th_mean'] + angular_shift
    fig2.add_trace(go.Scatter(x=E_ev_grid, y=np.degrees(exp_phase + ds['th_std']),
                              line=dict(color=c, width=0), showlegend=False, hoverinfo='skip'))
    fig2.add_trace(go.Scatter(x=E_ev_grid, y=np.degrees(exp_phase - ds['th_std']),
                              fill='tonexty', fillcolor=rgba,
                              line=dict(color=c, width=0), showlegend=False, hoverinfo='skip'))
    fig2.add_trace(go.Scatter(x=E_ev_grid, y=np.degrees(exp_phase),
                              mode='lines', name=ds['name'], line_color=c))

fig2.update_layout(title="Phase vs Energy", xaxis_title="Energy (eV)", yaxis_title="Phase (degrees)",
                   xaxis_range=[0.3, 0.9], yaxis_range=[0, 135])
fig2.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import colors


def add_banded_line(fig, x, y, dy, color, showlegend=True, **kwargs):
    fig.add_scatter(x=x, y=y + dy, line=dict(width=0, color=color),
                    showlegend=False, hoverinfo='skip', **kwargs)
    fig.add_scatter(x=x, y=y - dy, line=dict(width=0, color=color),
                    showlegend=False, hoverinfo='skip', fill='tonexty',
                    fillcolor=f"rgba{colors.to_rgba(color, alpha=0.3)}", **kwargs)
    fig.add_scatter(x=x, y=y, mode='lines', line=dict(width=2, color=color),
                    marker=dict(color=color, size=5), showlegend=showlegend, **kwargs)


fig = make_subplots(
        rows=1, cols=1,
        vertical_spacing=0.05,
        shared_xaxes=True,
).update_layout(
        height=400,
        width=600,
        margin=dict(t=0, r=0, b=0, l=0),
        template='simple_white+presentation',
        legend=dict(orientation="h", yanchor="top", y=1, xanchor="right", x=1)
)

for i, ds in enumerate(dataset_stats):
    exp_phase = ds['th_mean'] + angular_shift
    # add_banded_line(fig, E_ev_grid, 2 * (exp_phase + np.pi / 2), 2 * ds['th_std'], ds['color'],
    #                 row=1, col=1, name=ds['name'])
    add_banded_line(fig, E_ev_grid, ds['gd_mean'] * TIME_CONV_FS, ds['gd_std'] * TIME_CONV_FS, ds['color'],
                    row=1, col=1, name=ellipticities[i])

fig.update_xaxes(range=[0.4, 0.7], ticks="inside", ticklen=6, mirror=True)
fig.update_xaxes(row=1, title_text="Energy (eV)")
fig.update_yaxes(ticks="inside", ticklen=6, mirror=True)
# fig.update_yaxes(range=[4.5, 7], row=1, title_text="Phase Difference (rad)")
fig.update_yaxes(range=[0, 5], row=1, title_text="Group Delay (fs)", tickvals=[0, 1, 2, 3, 4, 5])

fig.add_hline(y=2.669, line_color='black', line_width=1, opacity=1, line_dash='dashdot',
              annotation_text="Laser Period", annotation_position="bottom right", row=1, col=1)

# fig.add_annotation(text="(a)", xref="paper", yref="y domain", x=0.025, y=0.975, showarrow=False)
# fig.add_annotation(text="(b)", xref="paper", yref="y2 domain", x=0.025, y=0.975, showarrow=False)
fig.show()
fig.write_image(r"C:\Users\mcman\DataspellProjects\vmi-analysis\Xenon Delay Figure Multiiple Ellipticities.png", scale=5)

In [ ]:
dataset = all_datasets[0]
I_exp = dataset['I']
r_vals = dataset['r_vals']
theta_vals = dataset['theta_vals']

ls = 25
hs = 110

df = extract_df(I_exp, r_vals, theta_vals, angle=45, e_scale=scale, sigma_small=ls, sigma_large=hs)

dog_fig = make_subplots(2, 1, shared_xaxes=True, vertical_spacing=0.05)
dog_fig.update_layout(height=600, width=600, margin=dict(t=0, r=0, b=0, l=0),
                      template='simple_white+presentation')
dog_fig.update_xaxes(title_text=r"$\phi~(rad)$", row=2)
dog_fig.update_yaxes(title_text=r"$p_r~(a.u.)$")

dog_fig.add_trace(go.Heatmap(x=theta_vals, y=r_vals * scale, z=I_exp, showscale=False,
                              colorscale=px.colors.sequential.Inferno))
dog_fig.add_trace(go.Heatmap(x=theta_vals, y=r_vals * scale,
                              z=skimage.filters.difference_of_gaussians(I_exp, ls, hs),
                              showscale=False, colorscale=px.colors.sequential.Inferno), row=2, col=1)
dog_fig.add_scatter(x=df['theta'], y=df['pr'] * scale, mode='lines', line_color="white", showlegend=False)
dog_fig.add_scatter(x=df['theta'], y=df['pr'] * scale, mode='lines', line_color="white", row=2, col=1,
                    showlegend=False)
dog_fig.add_annotation(text="(a)", xref="paper", yref="y domain", font_color="white",
                        x=0.025, y=0.975, showarrow=False)
dog_fig.add_annotation(text="(b)", font_color="white", xref="paper", yref="y2 domain",
                        x=0.025, y=0.975, showarrow=False)
dog_fig.write_image(r"C:\Users\mcman\DataspellProjects\vmi-analysis\Appendix Smoothing.png", scale=5)
dog_fig.show()